# CS 3110/3990: Data Privacy

## In-Class Exercise #4, week of 9/28/2026


In [ ]:
# Load the data and libraries
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt


def laplace_mech(v, sensitivity, epsilon):
    return v + np.random.laplace(loc=0, scale=sensitivity / epsilon)


def pct_error(orig, priv):
    return np.abs(orig - priv) / orig * 100.0


adult = pd.read_csv(
    "https://github.com/jnear/cs3110-data-privacy/raw/main/homework/adult_with_pii.csv"
)

## Question 1

For various values of $b$, write code to print out the percent error of summing the ages in the `adult` dataset,
clipped to each value of $b$.


In [ ]:
bs = range(1, 100, 10)
real_sum = adult["Age"].sum()

for b in bs:
    clipped_sum = adult["Age"].clip(lower=0, upper=b).sum()
    print("Percent error for", b, "is", pct_error(real_sum, clipped_sum))

# We don't want to pick a value too low, we want to be as close to the maximum
# of the dataset as possible.

Percent error for 1 is 97.4080940444511
Percent error for 11 is 71.48903448896206
Percent error for 21 is 46.000380495392264
Percent error for 31 is 25.509031989473492
Percent error for 41 is 11.7959939725709
Percent error for 51 is 4.329687317165198
Percent error for 61 is 1.1587597123836921
Percent error for 71 is 0.2341877497996031
Percent error for 81 is 0.03860675005194001
Percent error for 91 is 0.0


What value of $b$ is the best?


Any value of b larger than 81 is probably fine, because at b=81 the relative
error due to bias is small (0.03%). Values of b much larger than this risk
adding additional error due to variance when we add noise to achieve differential privacy.


## Question 2

For various values of $b$, print the result of a _differentially private_ sum of ages, clipped to each value of $b$. Use $\epsilon = 0.1$.


In [ ]:
bs = range(1, 200, 20)
real_sum = adult["Age"].sum()
print("number of candidates:", len(bs))

# This program satisfies differential privacy for a total privacy cost of
# epsilon = 10 * 0.1 by sequential composition (len(bs) * epsilon)
# epsilon=0.1 per query.
for b in bs:
    clipped_sum = adult["Age"].clip(lower=0, upper=b).sum()
    dp_sum = laplace_mech(clipped_sum, sensitivity=b, epsilon=0.1)
    print(
        "DP result for",
        b,
        "is",
        dp_sum,
        # "| error is",
        # pct_error(real_sum, dp_sum),
    )

# We want to minimize the total error, so by looking at various values of b and
# calculating the corresponding errors, we can choose the one that gives us the
# smallest error. however, looking at the percent error violates differential privacy.

# The error here now has some randomness due to the Laplace mechanism.

# One thing we could do is look for the first time when the value goes down./

number of candidates: 10
DP result for 1 is 32564.20098677947
DP result for 21 is 677946.7805290754
DP result for 41 is 1108762.7074456078
DP result for 61 is 1241732.7283875956
DP result for 81 is 1254844.5655832486
DP result for 101 is 1256403.8293304942
DP result for 121 is 1256668.3126095398
DP result for 141 is 1262097.4651173346
DP result for 161 is 1254570.5859401033
DP result for 181 is 1261921.4223855296


Which value of $b$ is the best now? Does it differ?


No it doesn't really differ. 81 or 91 (or even slightly higher) is still good.
The error due to bias is much greater than the error due to variance, until the
clipping parameter is really big.


## Question 3

Write an algorithm to _automatically pick the clipping parameter_ for the age column. Your algorithm should satisfy differential privacy.


In [33]:
def pick_b(epsilon):
    # Idea: try a bunch of different b values, (like question 2 prior)
    # and then find the first pair (b, b') where the noisy value decreases.
    # i.e. the noisy value for b is greater than the noisy value for b'.

    bs = range(1, 200, 20)
    # real_sum = adult["Age"].sum()

    # this forces the total privacy cost to be epsilon, without going over.
    epsilon_i = epsilon / len(bs)

    last_sum = 0

    # this satisfies len(bs) * epsilon_i for DP by sequential composition.
    for b in bs:
        clipped_sum = adult["Age"].clip(lower=0, upper=b).sum()
        dp_sum = laplace_mech(clipped_sum, sensitivity=b, epsilon=epsilon_i)
        if dp_sum < last_sum:
            return b
        else:
            last_sum = dp_sum

    # This is a safe catch to do:
    return bs[-1]  # if no decrease is found, return the last b value

    # Could also raise an exception:
    # raise Exception("No decrease found in dp_sum values.")


pick_b(1.0)

141

In [ ]:
# Total privacy cost is 1.0 + 2.0 = 3.0
b = pick_b(1.0)

result = laplace_mech(
    adult["Age"].clip(lower=0, upper=b).sum(), sensitivity=b, epsilon=2.0
)

In [ ]:
# TEST CASE for question 3

many_trials = np.mean([pick_b(1.0) for _ in range(100)])
assert many_trials > 70
assert many_trials < 500

## Question 4

What is the privacy cost of your algorithm, and why?


The privacy cost of pick_b is len(bs) \* epsilon_i = epsilon
Each iteration of the loop uses epsilon_i privacy cost, and there are len(bs) iterations.


## Question 5

Write code to generate a _histogram_ of education numbers in the `adult` dataset.


In [ ]:
def education_hist():
    # YOUR CODE HERE
    raise NotImplementedError()


education_hist()

In [ ]:
# TEST CASE for question 5
h = education_hist()
assert h["HS-grad"] == 10501
assert h["12th"] == 433
assert h["Doctorate"] == 413

## Question 6

Write code to release a _differentially private histogram_ of education numbers.


In [ ]:
def dp_education_hist(epsilon):
    # YOUR CODE HERE
    raise NotImplementedError()


dp_education_hist(1.0)

In [ ]:
# TEST CASE for question 6
h = dp_education_hist(1.0)
assert abs(h["HS-grad"] - 10501) < 100
assert abs(h["Doctorate"] - 413) < 100

## Question 7

What is the total privacy cost of `dp_education_hist`, and why?


YOUR ANSWER HERE


## Question 8

Write code to generate a differentially private _contingency table_ for the Education and Sex columns of the `adult` dataset.


In [ ]:
def dp_crosstab_education_sex(epsilon):
    # YOUR CODE HERE
    raise NotImplementedError()


dp_crosstab_education_sex(1.0)

In [ ]:
# TEST CASE for question 8
ct = dp_crosstab_education_sex(1.0)
assert abs(ct["Female"]["10th"] - 295) < 100
assert abs(ct["Male"]["10th"] - 638) < 100
assert abs(ct["Female"]["Bachelors"] - 1619) < 100

assert abs(ct["Female"]["10th"] - 295) > 0
assert abs(ct["Male"]["10th"] - 638) > 0
assert abs(ct["Female"]["Bachelors"] - 1619) > 0

## Question 9

Does parallel composition apply for the contingency table in question 1? Why or why not?


YOUR ANSWER HERE


## Question 10

Does the number of columns used in constructing the contingency table matter for privacy cost? Does it matter for accuracy?


YOUR ANSWER HERE
